In [2]:
import pandas as pd
import numpy as np

# Load raw datasets
appointments = pd.read_csv("../data/raw/apollo_appointments_fact.csv")
doctors = pd.read_csv("../data/raw/apollo_doctors_dim.csv")

print("Appointments:", appointments.shape)
print("Doctors:", doctors.shape)

Appointments: (75000, 52)
Doctors: (320, 15)


In [3]:
# Create working copies
cleaned_appointments = appointments.copy()
cleaned_doctors = doctors.copy()

# 1. Convert date columns
for col in ["appointment_date", "booking_date"]:
    cleaned_appointments[col] = pd.to_datetime(
        cleaned_appointments[col], errors="coerce"
    )

# 2. Remove duplicate records
cleaned_appointments = cleaned_appointments.drop_duplicates(
    subset="appointment_id"
)
cleaned_doctors = cleaned_doctors.drop_duplicates(
    subset="doctor_id"
)

# 3. Handle missing categorical values without changing original columns
cleaned_appointments["insurance_provider_clean"] = (
    cleaned_appointments["insurance_provider"]
    .fillna("Not Insured")
)

cleaned_appointments["patient_chronic_condition_clean"] = (
    cleaned_appointments["patient_chronic_condition"]
    .fillna("Unknown")
)

# 4. Feature engineering: calculate booking lead time
cleaned_appointments["booking_lead_days_calculated"] = (
    cleaned_appointments["appointment_date"]
    - cleaned_appointments["booking_date"]
).dt.days

# 5. Create additional useful features
cleaned_appointments["is_weekend"] = (
    cleaned_appointments["appointment_date"].dt.dayofweek >= 5
)

cleaned_appointments["lead_time_category"] = pd.cut(
    cleaned_appointments["booking_lead_days_calculated"],
    bins=[-np.inf, 0, 2, 7, 30, np.inf],
    labels=["Same day/Invalid", "1-2 days", "3-7 days",
            "8-30 days", "Above 30 days"]
)

# 6. Check important data quality issues
print("Appointment shape:", cleaned_appointments.shape)
print("Doctor shape:", cleaned_doctors.shape)

print("\nDuplicate appointment IDs:",
      cleaned_appointments["appointment_id"].duplicated().sum())

print("\nMissing values in important columns:")
print(cleaned_appointments[
    ["appointment_id", "patient_id", "doctor_id",
     "appointment_date", "booking_date"]
].isnull().sum())

print("\nMembership type missing values:")
print(cleaned_appointments["membership_type"].isna().sum())

print("\nInvalid booking lead times:")
print((cleaned_appointments["booking_lead_days_calculated"] < 0).sum())

# 7. Save cleaned datasets
cleaned_appointments.to_csv(
    "../data/processed/cleaned_appointments.csv", index=False
)

cleaned_doctors.to_csv(
    "../data/processed/cleaned_doctors.csv", index=False
)

print("\nDay 2 cleaning completed!")

Appointment shape: (75000, 57)
Doctor shape: (320, 15)

Duplicate appointment IDs: 0

Missing values in important columns:
appointment_id      0
patient_id          0
doctor_id           0
appointment_date    0
booking_date        0
dtype: int64

Membership type missing values:
58225

Invalid booking lead times:
0

Day 2 cleaning completed!
